In [ ]:
import torch
import torch.nn as nn
import numpy as np
from PIL import Image

from module import GlobalSemanticBranch,LocalDetailBranch,FrequencyDomainBranch,CrossAttentionFusion
from module import SwinTransformerBlock,SwinTransformerEncoder


/home/liangshuqiao/hong/deepfake/rine_opt/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from tools.image_preprocess import transforms_train
img_path = 'dataset/Celeb-DF/00000/00000_frame000000.jpg'

img_data = Image.open(img_path)
img = transforms_train(img_data)

In [4]:
img_size = 224
scale = 0.5
depths = [2,2,6,2]
embed_dim = 96
num_heads = [3,6,12,24]
window_size = 7

swin = SwinTransformerEncoder(
                img_size=int(img_size * scale),
                embed_dim=embed_dim,
                depths=depths,
                num_heads=num_heads,
                window_size=window_size
            )
swin

/home/liangshuqiao/hong/deepfake/rine_opt/env/lib/python3.10/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


SwinTransformerEncoder(
  (model): SwinTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0): BasicLayer(
        (blocks): ModuleList(
          (0): SwinTransformerBlock(
            (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=96, out_features=288, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=96, out_features=96, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (drop_path): Identity()
            (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=96, out_features=384, bias=True)
              (act): GELU(a

In [10]:
class MultiScaleHierarchicalTransformer(nn.Module):
    """
    多尺度层次化Transformer架构
    
    设计理念：
    - 多尺度输入：处理不同分辨率的图像
    - 层次化特征：从局部到全局的特征提取
    - 多分支融合：结合语义、细节和频域信息
    """
    
    def __init__(self, num_classes=2, img_size=224, embed_dim=96, depths=[2, 2, 6, 2], 
                 num_heads=[3, 6, 12, 24], window_size=7, use_scales=[1,0.5, 0.25]):
        super().__init__()
        self.img_size = img_size
        self.use_scales = use_scales # 1,0.5,0.25

        self.scales_encoder = nn.ModuleList() # 
        for scale in use_scales:
            encoder = SwinTransformerEncoder(
                img_size=int(img_size * scale),
                embed_dim=embed_dim,
                depths=depths,
                num_heads=num_heads,
                window_size=window_size
            )
            # print(scale)
            self.scales_encoder.append(encoder)

        
        


    def forward(self,x):
        pass

model = MultiScaleHierarchicalTransformer()
model

1
0.5
0.25


MultiScaleHierarchicalTransformer(
  (scales_encoder): ModuleList(
    (0-2): 3 x SwinTransformerEncoder(
      (model): SwinTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
          (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        )
        (pos_drop): Dropout(p=0.0, inplace=False)
        (layers): ModuleList(
          (0): BasicLayer(
            (blocks): ModuleList(
              (0): SwinTransformerBlock(
                (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
                (attn): WindowAttention(
                  (qkv): Linear(in_features=96, out_features=288, bias=True)
                  (attn_drop): Dropout(p=0.0, inplace=False)
                  (proj): Linear(in_features=96, out_features=96, bias=True)
                  (proj_drop): Dropout(p=0.0, inplace=False)
                )
                (drop_path): Identity()
                (norm2): LayerNorm((96,), ep